# Circle tracking under process noise

Compare vanilla, tube and robust MPPI on a planar double integrator. Every run receives the same acceleration disturbances. The seven configurations follow the [MPPI-Generic CoRL 2020 example](https://github.com/ACDSLab/MPPI-Generic/blob/abcd1ab419cb27efa113573932adbde1e0380835/examples/double_integrator_CORL2020.cu).

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation. See LICENSE.

Authors: Frank Dellaert and Akash Ratheesh

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam_unstable/examples/planning/DoubleIntegratorCoRL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam

The state contains planar position and velocity. The controls are accelerations. The nominal trajectory runs counterclockwise on a circle of radius 2 at speed 2. The CPU demonstration uses a shorter rollout, fewer samples and fewer decisions than the CUDA experiment, which used 50 steps, 1,024 samples and 5,000 decisions. Increasing these three settings changes the numerical result and runtime.

In [3]:
H, samples, decisions = 10, 24, 12
dt = 0.02
X = [gtsam.symbol('x', k) for k in range(H+1)]
U = [gtsam.symbol('u', k) for k in range(H)]

def dynamics(x, u):
    return np.r_[x[2:], u]

def transition(x, u):
    return x + dt * dynamics(x, u)

def values(start, controls):
    v = gtsam.Values()
    x = start.copy()
    for k in range(H):
        v.insert(X[k], x)
        v.insert(U[k], controls[:, k].copy())
        x = transition(x, controls[:, k])
    v.insert(X[-1], x)
    return v

def controls(v):
    return np.column_stack([v.atVector(k) for k in U])

def shift(v):
    u = controls(v)
    return np.column_stack((u[:, 1:], u[:, -1]))

The standard cost penalizes leaving the annulus, speed error and angular-momentum error. The robust cost uses the upstream device's continuous radial penalty and squared speed and angular-momentum errors. Each unary factor converts a nonnegative stage cost into a residual so that GTSAM's squared error equals the stage cost divided by the horizon. The terminal cost is zero.

In [4]:
def track_graph(robust_cost):
    graph = gtsam.NonlinearFactorGraph()
    def residual(factor, v, jacobians):
        x, y, vx, vy = v.atVector(factor.keys()[0])
        radius = np.hypot(x, y)
        speed_error = np.hypot(vx, vy) - 2.
        momentum_error = x * vy - y * vx - 4.
        if robust_cost:
            cost = 100. * min(1., abs(radius - 2.) / 0.125)
            cost += speed_error**2 + momentum_error**2
        else:
            cost = 1000. * (radius < 1.875 or radius > 2.125)
            cost += abs(speed_error) + abs(momentum_error)
        return np.array([np.sqrt(2. * cost / H)])
    for key in X[:-1]:
        graph.add(gtsam.CustomFactor(gtsam.noiseModel.Unit.Create(1), [key], residual))
    return graph

# Upstream integrates the running precisions over time and uses identity terminal precision.
# The linear transition yields the same tracking gains along every reference.
q = np.diag([500., 500., 100., 100.])
ddp = gtsam.DDPTracking(dt * q, dt * np.eye(2), np.eye(4))
ddp.optimize(np.zeros(4), np.zeros((4, H+1)), np.zeros((2, H)), transition, 1)
gains = [ddp.gain(k) for k in range(H)]
def feedback(k, actual, nominal, u):
    return gains[k] @ (actual - nominal)

The tube update compares minimum sampled costs from nominal and measured starts. Robust MPPI first selects a nominal starting state by free energy, then includes feedback in its sampled real-system rollouts. The first command from each returned actual trajectory includes the tracking correction.

Each method smooths the final open-loop control sequence with the five-point quadratic filter from MPPI-Generic. The simulation supplies the preceding two nominal and applied controls before each optimization.

In [5]:
def run(method, variance, robust_cost, sigma, beta, noise_seed, proposal_seed):
    disturbances = np.random.default_rng(noise_seed).standard_normal((decisions, 2))
    planner = gtsam.SamplingPlannerVectorVector(X, U, dt)
    tracking = gtsam.MPPITracking(sigma**2 * np.eye(2), samples, beta, proposal_seed)
    graph = track_graph(robust_cost)
    x = np.array([2., 0., 0., 1.])
    nominal = actual = values(x, np.zeros((2, H)))
    history = [x.copy()]
    nominal_inputs = np.zeros((2, 2))
    measured_inputs = np.zeros((2, 2))
    for tick, disturbance in enumerate(disturbances):
        tracking.setControlHistory(nominal_inputs, measured_inputs)
        start = values(x, shift(actual))
        if method == "Vanilla":
            # A colored Gaussian with exponent zero is a white Gaussian.
            actual = nominal = tracking.colored(planner, graph, start, dynamics, 0., 1)
        elif method == "Tube":
            reference = values(nominal.atVector(X[int(tick > 0)]), shift(nominal))
            result = tracking.tube(planner, graph, reference, start, dynamics, feedback, 2. if robust_cost else 20., 1)
            nominal, actual = result.nominal, result.actual
        else:
            result = tracking.robust(planner, graph, nominal, start, dynamics, feedback, 20., 5,
                                    int(tick > 0), 1, np.eye(2))
            nominal, actual = result.nominal, result.actual
        nominal_inputs = np.column_stack((nominal_inputs[:, -1], nominal.atVector(U[0])))
        measured_inputs = np.column_stack((measured_inputs[:, -1], actual.atVector(U[0])))
        x = transition(x, measured_inputs[:, -1])
        x[2:] += dt * np.sqrt(variance) * disturbance
        history.append(x.copy())
    return np.array(history)

### Selecting a spread and temperature for noisy tracking

We tune on disturbed closed-loop trajectories because Tube and Robust MPPI use feedback and a nominal reference. The score is the mean squared Cartesian position error against the moving circular target at each decision. It penalizes a controller that stays still on the annulus. For each method and cost graph, we choose the lowest median score at disturbance variance 100 on four paired process and proposal seeds. The evaluation uses five separate pairs. The vanilla, tube, and robust runs keep the same number of decisions and samples for every setting of a given method. Robust also evaluates initial-state candidates and tracked rollouts, so the methods have different total costs in computation.

In [6]:
from itertools import product

angles = dt * np.arange(1, decisions + 1)  # speed 2 divided by radius 2
target_positions = 2 * np.column_stack((np.cos(angles), np.sin(angles)))

def tracking_error(method, robust_cost, setting, seed_pairs):
    errors = []
    for noise_seed, proposal_seed in seed_pairs:
        history = run(method, 100., robust_cost, *setting, noise_seed, proposal_seed)
        errors.append(np.mean(np.sum((history[1:, :2] - target_positions)**2, axis=1)))
    return np.median(errors)

calibration = list(zip(range(100, 104), range(200, 204)))
evaluation = list(zip(range(7, 12), range(42, 47)))
selected = {}
for method, robust_cost in product(("Vanilla", "Tube", "Robust"), (False, True)):
    settings = product((0.5, 1., 2.), (0.125, 0.5, 2.))
    selected[method, robust_cost] = min(settings, key=lambda setting:
        tracking_error(method, robust_cost, setting, calibration))

In [7]:
print("Method   Cost       Sigma   Beta    Before       Tuned  (median squared position error, m²)")
for (method, robust_cost), setting in selected.items():
    before = tracking_error(method, robust_cost, (1., 0.5), evaluation)
    after = tracking_error(method, robust_cost, setting, evaluation)
    print(f"{method:8s} {'robust' if robust_cost else 'standard':9s} "
          f"{setting[0]:4.1f} {setting[1]:6.3f} {before:10.6f} {after:10.6f}")

Method   Cost       Sigma   Beta    Before       Tuned  (median squared position error, m²)


Vanilla  standard   2.0  2.000   0.027718   0.023726


Vanilla  robust     2.0  2.000   0.026743   0.017925


Tube     standard   2.0  2.000   0.027997   0.023647


Tube     robust     2.0  2.000   0.022424   0.015861


Robust   standard   2.0  2.000   0.027536   0.022918


Robust   robust     2.0  2.000   0.023367   0.017924


In [8]:
runs = {}
for method, variance, robust_cost in [
    ("Vanilla", 1., False), ("Vanilla", 100., False), ("Vanilla", 100., True),
    ("Tube", 100., False), ("Tube", 100., True), ("Robust", 100., False), ("Robust", 100., True)
]:
    label = f"{method}, variance {variance:g}, {'robust' if robust_cost else 'standard'} cost"
    runs[label] = run(method, variance, robust_cost, *selected[method, robust_cost], 7, 42)
    radius = np.linalg.norm(runs[label][:, :2], axis=1)
    print(f"{label:44s}  maximum radial error {np.max(np.abs(radius-2)):.4f}")
assert all(np.isfinite(x).all() for x in runs.values())

Vanilla, variance 1, standard cost            maximum radial error 0.0070
Vanilla, variance 100, standard cost          maximum radial error 0.0558


Vanilla, variance 100, robust cost            maximum radial error 0.0366


Tube, variance 100, standard cost             maximum radial error 0.0538


Tube, variance 100, robust cost               maximum radial error 0.0386


Robust, variance 100, standard cost           maximum radial error 0.0586


Robust, variance 100, robust cost             maximum radial error 0.0362


The plot compares radial errors for identical disturbances and decision times. The dashed bounds mark the planning annulus. Interactive charts require an internet connection.

In [9]:
import plotly.graph_objects as go
from IPython.display import HTML, display

fig = go.Figure()
for label, states in runs.items():
    fig.add_scatter(x=dt*np.arange(decisions+1), y=np.linalg.norm(states[:, :2], axis=1)-2.,
                    mode='lines', name=label)
for bound in [-0.125, 0.125]:
    fig.add_hline(y=bound, line_dash='dash', line_color='gray')
fig.update_layout(xaxis_title='Time (s)', yaxis_title='Radial error (m)')
from html import escape

document = fig.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Circle tracking under process noise" style="width:100%;border:0" '
             f'height="{fig.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead

